In [ ]:
!pip install "datasets<3.0.0"

In [ ]:
!pip install evaluate

In [3]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.1 MB/s eta 0:00:00


In [4]:
import random
import evaluate
from datasets import load_dataset

In [5]:
def model_random_baseline(train_ds, test_ds):
  total = len(train_ds)
  preds = []
  for i in range(len(test_ds)):
    random_index = random.randint(0, total-1)
    preds.append(train_ds[random_index]["transcription"])
  return preds

##Baseline for Sindhi

In [6]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
sindhi_ds = load_dataset("google/fleurs", "sd_in", trust_remote_code=True)

In [ ]:
predictions = model_random_baseline(sindhi_ds["train"], sindhi_ds["test"])
references = [ sindhi_ds["test"][i]["transcription"] for i in range(len(sindhi_ds["test"])) ]

In [ ]:
wer = wer_metric.compute(predictions=predictions, references=references)
cer = cer_metric.compute(predictions=predictions, references=references)

In [ ]:
print(f"{'Sindhi (sd_in)':<15} | {wer*100:>9.2f}% | {cer*100:>9.2f}%")

Sindhi (sd_in)  |    112.48% |     87.45%


##Baseline for Akan

In [ ]:
akan_ds = load_dataset(
    "google/WaxalNLP",
    "aka_asr",
    split=["train", "validation", "test"],
    verification_mode="no_checks"
)

In [9]:
ds = load_dataset("google/WaxalNLP", "aka_asr", split="train", streaming=True)

# 2. Collect just the text labels into a list (the "pool")
# We use a loop to 'drain' the text from the stream
print("Collecting transcriptions for the baseline pool...")
transcription_pool = []
for example in ds:
    transcription_pool.append(example["transcription"])

print(f"Pool created with {len(transcription_pool)} sentences.")

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Pool created with 10107 sentences.


In [10]:
test_ds = load_dataset("google/WaxalNLP", "aka_asr", split="test", streaming=True)
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

references = []
predictions = []

print("Running random baseline on test set...")
for example in test_ds:
    references.append(example["transcription"])
    # The 'Naive' part: pick a random sentence from our pool
    predictions.append(random.choice(transcription_pool))

# Calculate Word Error Rate
wer = wer_metric.compute(predictions=predictions, references=references)
cer = cer_metric.compute(predictions=predictions, references=references)
print(f"{'Akan (aka_asr)':<15} | {wer*100:>9.2f}% | {cer*100:>9.2f}%")

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Running random baseline on test set...
Akan (aka_asr)  |    111.08% |     84.53%
